# Module 4: Model Context Protocol (MCP)

**Course:** UNDP IICPSD Agentic AI & Vibe Coding Training Series 2026 | **Instructor:** Phyo Ko Ko, Senior Cyber Security Analyst

---

## Why This Module Matters

Traditional AI coding assistants are **sandboxed** - they can only see the files in your workspace. They cannot query a live database, read from a remote filesystem, call an external API, or send a notification. Every external connection required hardcoded, custom integration code.

**MCP (Model Context Protocol)** is an open standard created by Anthropic that solves this. It allows any AI agent to connect to any external tool or data source through a single standardized interface - without custom integration code for each tool.

Think of it like a **USB port**: before USB, every device needed its own proprietary cable and connector. USB created one universal standard that any device could plug into. MCP does the same for AI agents and external tools.

```
[ User Request ]
       |
       v
[ MCP Client / AI Agent ]  <-- discovers what tools are available
       |
       v
[ MCP Server ]  <-- hosts registered tools with descriptions
  |         |         |
  v         v         v
[DB]    [Files]   [APIs / Notifications]
```

## How It Works

1. **MCP Server** - a process that registers tools (functions) with their names, descriptions, and parameter schemas
2. **MCP Client / Agent** - connects to the server, discovers available tools, and decides which tool to call based on user intent
3. **Dynamic Routing** - the agent reads the tool descriptions and routes requests to the right tool automatically


## Key Terms

| Term | Definition |
|---|---|
| **MCP Server** | A process that registers and exposes tools (functions) to any connected AI agent. |
| **MCP Client** | The AI agent that connects to an MCP server, discovers available tools, and calls them. |
| **Tool Registration** | The process of declaring a tool's name, description, and parameters on the MCP server. |
| **Tool Discovery** | The agent reading the server's tool list to understand what capabilities are available. |
| **Dynamic Routing** | The agent deciding which tool to call based on the user request and tool descriptions. |
| **Sandbox** | The restricted environment an AI agent operates in - MCP is what allows it to break out safely. |


---

## Lab 4: Connecting Agents to the Real World with MCP

### Objective

Simulate a full MCP architecture in Python. We build an MCP Server that registers tools, and an MCP Agent (Client) that connects to the server, discovers the available tools, and dynamically routes user requests to the correct tool.


In [ ]:
# --- MCP SERVER ---
class MCPServer:
    def __init__(self, name):
        self.name = name
        self.tools = {}

    def register_tool(self, name, description, parameters, handler):
        self.tools[name] = {
            "name": name,
            "description": description,
            "parameters": parameters,
            "handler": handler
        }
        print(f"  [MCP Server] Tool registered: '{name}'")

    def list_tools(self):
        return {name: t["description"] for name, t in self.tools.items()}

    def call_tool(self, tool_name, **kwargs):
        if tool_name not in self.tools:
            return f"Error: Tool '{tool_name}' not found on server."
        return self.tools[tool_name]["handler"](**kwargs)


# --- TOOL DEFINITIONS ---
def query_database(table, condition):
    mock_db = {
        "users": [
            {"id": 1, "name": "Alice", "email": "alice@undp.org", "role": "Admin"},
            {"id": 2, "name": "Bob",   "email": "bob@undp.org",   "role": "Analyst"}
        ]
    }
    results = [row for row in mock_db.get(table, []) if condition.lower() in str(row).lower()]
    return results if results else "No records found."

def read_file(filepath):
    mock_files = {
        "/config/settings.json": '{"debug": false, "version": "2.1.0"}',
        "/logs/access.log": "2026-07-07 10:23 - User alice logged in\n2026-07-07 10:25 - Query executed"
    }
    return mock_files.get(filepath, f"Error: File '{filepath}' not found.")

def send_notification(channel, message):
    return f"[Notification sent to {channel}]: {message}"


# --- REGISTER TOOLS ON THE SERVER ---
print("=== Booting MCP Server ===")
server = MCPServer("UNDP-Enterprise-MCP-Server")
server.register_tool("query_database",
                     "Query a database table with a search condition",
                     {"table": "string", "condition": "string"},
                     query_database)
server.register_tool("read_file",
                     "Read the contents of a file from the local filesystem",
                     {"filepath": "string"},
                     read_file)
server.register_tool("send_notification",
                     "Send a message to a Slack channel or email address",
                     {"channel": "string", "message": "string"},
                     send_notification)


# --- MCP CLIENT / AI AGENT ---
class MCPAgent:
    def __init__(self, server):
        self.server = server
        print(f"\n[Agent] Connected to: {server.name}")
        print("[Agent] Tools discovered:")
        for name, desc in server.list_tools().items():
            print(f"   - {name}: {desc}")
        print()

    def handle_request(self, user_request):
        print(f"[Agent] Request: '{user_request}'")
        req = user_request.lower()

        if any(w in req for w in ["database", "query", "user", "find"]):
            print("[Agent] Routing to -> query_database")
            result = self.server.call_tool("query_database",
                                           table="users",
                                           condition=user_request.split()[-1])
        elif any(w in req for w in ["file", "read", "log", "config"]):
            print("[Agent] Routing to -> read_file")
            filepath = "/logs/access.log" if "log" in req else "/config/settings.json"
            result = self.server.call_tool("read_file", filepath=filepath)
        elif any(w in req for w in ["notify", "send", "alert", "message"]):
            print("[Agent] Routing to -> send_notification")
            result = self.server.call_tool("send_notification",
                                           channel="#ops-alerts",
                                           message=user_request)
        else:
            result = "No suitable MCP tool found for this request."

        print(f"[Agent] Result: {result}\n")
        return result


# --- RUN THE AGENT ---
agent = MCPAgent(server)

print("=== Live Agent Requests ===")
agent.handle_request("Find all users named Alice")
agent.handle_request("Read the access log file")
agent.handle_request("Send alert: Security scan completed successfully")


## What Just Happened?

1. **Server boot** - 3 tools were registered with their names, descriptions, and handler functions
2. **Agent connection** - the agent connected and read the full tool list (tool discovery)
3. **Dynamic routing** - for each request, the agent matched keywords to pick the right tool and called it through the server

Notice that the agent never needed hardcoded logic for each specific service. It discovered the tools dynamically from the server. In production MCP, the routing uses the **tool descriptions** and an LLM to choose the best tool - far more accurate than keyword matching.


---

## Lab 4b: Multi-Agent Pipelines

### Why This Follows from MCP

In Lab 4 you built a **single agent** that discovers and calls tools on an MCP server. The natural next step is **multi-agent orchestration**: instead of one agent doing everything, you coordinate a **pipeline of specialized agents** — each expert at one job — with human approval gates controlling every handoff.

### The Architecture

```
Feature Spec
     |
     v
[Agent 1: Product Owner]  — breaks spec into structured task + acceptance criteria
     | human gate
     v
[Agent 2: Coder]          — reads task, generates modular code
     | human gate
     v
[Agent 3: Tester]         — writes & runs unit tests, reports failures
     | human gate (only if tests pass)
     v
[Agent 4: Auditor]        — security compliance review
     | human gate
     v
   Deploy
```

### Objective

Simulate this full pipeline in Python. Run it with a feature spec and observe how each agent processes the output of the previous one, with human approval gates between stages.

In [ ]:
# ============================================================
# Lab 4b: Multi-Agent Pipeline Simulation
# ============================================================

class ProductOwnerAgent:
    """Agent 1: Breaks a feature spec into a structured task file."""
    def process(self, feature_spec):
        print("\n[Agent 1: Product Owner] Processing feature spec...")
        task = {
            "feature": feature_spec,
            "acceptance_criteria": [
                "Must accept a username string as input",
                "Must return email and role fields for valid users",
                "Must return a clear error message for unknown users",
                "Must handle empty or invalid input gracefully",
                "Must not expose raw database errors to the caller"
            ],
            "security_requirements": [
                "No hardcoded credentials",
                "Sanitize and validate all input",
                "Use parameterized lookups only"
            ]
        }
        print(f"[Agent 1] Done — {len(task['acceptance_criteria'])} acceptance criteria defined")
        return task


class CoderAgent:
    """Agent 2: Reads the task file and writes modular code."""
    def process(self, task):
        print(f"\n[Agent 2: Coder] Generating code for: {task['feature']}")
        code = '''\
def get_user_profile(username: str) -> dict:
    """Look up a user profile by username. Returns email and role."""
    if not username or not isinstance(username, str):
        return {"error": "Invalid username — must be a non-empty string"}
    username = username.strip().lower()
    mock_db = {
        "alice": {"email": "alice@undp.org", "role": "Admin"},
        "bob":   {"email": "bob@undp.org",   "role": "Analyst"},
        "chen":  {"email": "chen@undp.org",  "role": "Developer"},
    }
    if username not in mock_db:
        return {"error": f"User '{username}' not found"}
    return {"username": username, **mock_db[username]}
'''
        print(f"[Agent 2] Done — {len(code.strip().splitlines())} lines generated")
        return code


class TesterAgent:
    """Agent 3: Writes and runs unit tests against the generated code."""
    def process(self, code, task):
        print("\n[Agent 3: Tester] Running unit tests...")
        exec_scope = {}
        exec(code, exec_scope)
        fn = exec_scope["get_user_profile"]

        tests = [
            ("Valid user alice returns email",  lambda: fn("alice"),   lambda r: r.get("email") == "alice@undp.org"),
            ("Valid user bob role check",       lambda: fn("BOB"),     lambda r: r.get("role") == "Analyst"),
            ("Case-insensitive lookup",         lambda: fn("CHEN"),    lambda r: "error" not in r),
            ("Unknown user returns error dict", lambda: fn("unknown"), lambda r: "error" in r),
            ("Empty string is rejected",        lambda: fn(""),        lambda r: "error" in r),
            ("None input is rejected",          lambda: fn(None),      lambda r: "error" in r),
        ]

        passed = failed = 0
        for name, run, check in tests:
            try:
                ok = check(run())
                if ok: passed += 1
                else:  failed += 1
                print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
            except Exception as e:
                failed += 1
                print(f"  [ERROR] {name}: {e}")

        print(f"\n[Agent 3] Done — {passed} passed, {failed} failed")
        return {"passed": passed, "failed": failed}


class AuditorAgent:
    """Agent 4: Reviews code for security compliance."""
    def process(self, code):
        import re
        print("\n[Agent 4: Auditor] Running security review...")
        checks = [
            ("No hardcoded secrets",        not bool(re.search(r'password\s*=\s*["\'].+["\']', code, re.I))),
            ("No eval() usage",             "eval(" not in code),
            ("Input type validation",       "isinstance" in code),
            ("Input sanitisation present",  ".strip()" in code),
            ("Error path for unknown user", "not found" in code.lower()),
        ]
        all_clear = all(ok for _, ok in checks)
        for name, ok in checks:
            print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
        verdict = "APPROVED" if all_clear else "REJECTED — fix security issues before deployment"
        print(f"\n[Agent 4] Verdict: {verdict}")
        return {"approved": all_clear, "verdict": verdict}


def human_gate(stage, summary):
    """Simulates a human orchestrator approving a handoff between agents."""
    print(f"\n{'─' * 55}")
    print(f"  HUMAN APPROVAL GATE — {stage}")
    print(f"  {summary}")
    print(f"  Decision: APPROVED [simulated]")
    print(f"{'─' * 55}")


# ─── Run the full pipeline ──────────────────────────────────
print("=" * 55)
print("  MULTI-AGENT PIPELINE: Feature Build Request")
print("=" * 55)

spec = "Build a user profile lookup function: accepts username, returns email and role"

# Stage 1 — Product Owner
po   = ProductOwnerAgent()
task = po.process(spec)
human_gate("Agent 1 → Agent 2", f"{len(task['acceptance_criteria'])} criteria defined")

# Stage 2 — Coder
coder = CoderAgent()
code  = coder.process(task)
human_gate("Agent 2 → Agent 3", "Code ready for testing")

# Stage 3 — Tester
tester       = TesterAgent()
test_results = tester.process(code, task)
if test_results["failed"] > 0:
    print("\n[Pipeline] Tests FAILED — code loops back to Coder (not simulated here)")
else:
    human_gate("Agent 3 → Agent 4", f"All {test_results['passed']} tests passing")

# Stage 4 — Auditor
auditor = AuditorAgent()
audit   = auditor.process(code)
human_gate("Agent 4 → Deploy", audit["verdict"])

# Summary
print("\n" + "=" * 55)
print("  PIPELINE SUMMARY")
print("=" * 55)
total = test_results["passed"] + test_results["failed"]
print(f"  Feature : {spec[:52]}...")
print(f"  Tests   : {test_results['passed']} / {total} passing")
print(f"  Security: {audit['verdict']}")
print("=" * 55)

## What Just Happened? (Lab 4b)

1. **Product Owner Agent** parsed the plain-English spec and produced a structured task with explicit acceptance criteria and security requirements — ready to pass to any coding agent.

2. **Coder Agent** consumed the task and generated working Python code. In a real vibe coding workflow, this would be Claude Sonnet writing to your actual file.

3. **Tester Agent** wrote 6 unit tests covering normal cases, edge cases (case sensitivity), and rejection cases (empty/None input) — then executed them directly. No manual test writing required.

4. **Auditor Agent** ran 5 security checks against the generated code: secret hygiene, eval() absence, input type validation, sanitisation, and error path coverage.

5. **Human gates** appear between every stage. As the orchestrator, you review the summary at each gate before approving the next stage. You do not write a single line of code.

> **Key insight:** MCP from Lab 4 is what enables this in production. The Coder Agent retrieves your actual codebase via RAG, the Tester runs tests via a terminal MCP tool, and the Auditor calls TruffleHog and Bandit via security MCP tools — all without you manually doing any of it.

---

## Assignment 4

### Part A: Reflection Questions

**Q1.** Before MCP existed, how did AI agents interact with external services like databases or APIs? What was the main problem with that approach?

> *Your answer here...*

**Q2.** In Lab 4, the agent uses keyword matching to decide which tool to call. In a real production AI agent, what more sophisticated method would be used for tool selection? Why is it more reliable?

> *Your answer here...*

**Q3.** MCP is described as a USB port for AI agents. Explain this analogy - what does the USB port represent, what is the device, and what is the computer?

> *Your answer here...*

### Part B: Coding Exercise - Add a New Tool

Add a 4th tool called `check_system_status` to the MCP server that:
1. Accepts a `service_name` parameter (e.g. `'database'`, `'api'`, `'auth'`)
2. Returns a mock status report for at least 3 different service names
3. Is registered on the server
4. Is routed by the agent when the request contains `'status'`, `'health'`, or `'check'`

### Part D: Multi-Agent Design Exercise

You are the human orchestrator managing the pipeline from Lab 4b.

**Q5.** The Tester Agent runs 6 tests and 2 fail. As the orchestrator at the human approval gate between Agent 3 and Agent 4, what do you do? Select and explain your choice:

- A) Approve anyway — 4/6 is good enough
- B) Reject and send the failing test output back to the Coder Agent to fix
- C) Skip the Auditor and go directly to deployment
- D) Rewrite the failing tests yourself

> *Your answer here...*

**Q6.** Design a 5th agent for the pipeline. What specialization does it have, what does it take as input, what does it produce as output, and at what stage does it sit?

| Property | Your Design |
|---|---|
| **Agent name** | |
| **Specialization** | |
| **Input** | |
| **Output** | |
| **Position in pipeline** | |

**Q7.** The multi-agent pipeline and the single MCP agent from Lab 4 both involve AI agents calling tools. What is the fundamental difference in their architecture?

> *Your answer here...*

In [ ]:
# TODO 1: Define check_system_status
def check_system_status(service_name):
    statuses = {
        # Add at least 3 services, for example:
        # "database": "Online - response time 12ms, last backup 2 hours ago",
        # "api": "Online - 99.8% uptime last 30 days",
        # "auth": "Online - 0 failed login attempts in last hour",
    }
    return statuses.get(service_name.lower(), f"Unknown service: '{service_name}'")


# Rebuild the server with all 4 tools
print("=== Extended MCP Server ===")
server2 = MCPServer("UNDP-Extended-MCP-Server")
server2.register_tool("query_database",
                      "Query a database table with a search condition",
                      {"table": "string", "condition": "string"}, query_database)
server2.register_tool("read_file",
                      "Read the contents of a file from the local filesystem",
                      {"filepath": "string"}, read_file)
server2.register_tool("send_notification",
                      "Send a message to a Slack channel or email address",
                      {"channel": "string", "message": "string"}, send_notification)

# TODO 2: Register check_system_status on server2


# TODO 3: Extend the agent to route status/health/check requests
class ExtendedMCPAgent(MCPAgent):
    def handle_request(self, user_request):
        req = user_request.lower()
        if any(w in req for w in ["status", "health", "check"]):
            print(f"[Agent] Request: '{user_request}'")
            print("[Agent] Routing to -> check_system_status")
            service = user_request.split()[-1]
            result = self.server.call_tool("check_system_status", service_name=service)
            print(f"[Agent] Result: {result}\n")
            return result
        return super().handle_request(user_request)


agent2 = ExtendedMCPAgent(server2)

# TODO 4: Test with a status request and one existing request
agent2.handle_request("Check the database status")
agent2.handle_request("Find all users named Bob")


### Part C: Design Exercise

You are advising a UNDP country office that wants to build an agentic AI assistant for their operations team.

Design **3 MCP tools** they would need. Fill in the table below:

| Tool Name | Description | Parameters | Real-world system it connects to |
|---|---|---|---|
| `tool_1_name` | What it does | param1, param2 | e.g. SharePoint, SAP, Gmail |
| `tool_2_name` | What it does | param1 | e.g. UN Data Portal |
| `tool_3_name` | What it does | param1, param2 | e.g. Internal HR system |

*Edit this table with your own tool designs.*

**Follow-up:** What security controls would you put in place before allowing an AI agent to call these tools autonomously?

> *Your answer here...*
